# project_3_pr_dm_2.ipynb

- 목적: 1차 정리된 지진/해일 대피소 CSV에서 주소를 분해해 시도·시군구를 붙이고 프로젝트 대상 권역만 남긴다.
- 입력: `earthquake_shelter_clean.csv`, `tsunami_shelter_clean.csv`
- 출력: `earthquake_shelter_clean_2.csv`, `tsunami_shelter_clean_2.csv`
- 메모: 추천 페이지가 지역 필터를 걸 수 있도록 주소 기반 행정구역 컬럼을 만드는 노트북이다.


## 1. 지진 대피소 주소 정리
정제된 지진 대피소 CSV를 읽고 주소에서 시도/시군구를 분해한다.


In [ ]:
import pandas as pd

eq = pd.read_csv("data/earthquake_shelter_clean.csv", encoding="utf-8-sig")

eq.head()


In [ ]:
eq.columns


In [ ]:
# 주소 문자열에서 시도와 시군구를 분해하기 위한 함수다.
def clean_korea_address(address):

    if pd.isna(address):
        return None, None

    parts = address.split()

    sido_map = {
        "서울특별시":"서울",
        "부산광역시":"부산",
        "대구광역시":"대구",
        "인천광역시":"인천",
        "광주광역시":"광주",
        "대전광역시":"대전",
        "울산광역시":"울산",
        "세종특별자치시":"세종",

        "경기도":"경기",
        "강원도":"강원",
        "강원특별자치도":"강원",

        "충청북도":"충북",
        "충청남도":"충남",

        "전라북도":"전북",
        "전북특별자치도":"전북",

        "전라남도":"전남",

        "경상북도":"경북",
        "경상남도":"경남",

        "제주특별자치도":"제주"
    }

    parts = address.split()

    sido = parts[0] if len(parts) > 0 else None
    sigungu = parts[1] if len(parts) > 1 else None

    if sido in sido_map:
        sido = sido_map[sido]

    return sido, sigungu


In [ ]:
eq[["시도","시군구"]] = (
    eq["주소"]
    .apply(lambda x: pd.Series(clean_korea_address(x)))
)


## 2. 동남권 대상 지역만 남기기
현재 프로젝트 범위인 경남·경북·울산·대구·부산만 남겨 지진 전용 CSV를 저장한다.


In [ ]:
target_region = ["경남","경북","울산","대구","부산"]

eq = eq[eq["시도"].isin(target_region)]


In [ ]:
eq["시도"].value_counts()


In [ ]:
eq.to_csv(
    "data/earthquake_shelter_clean_2.csv",
    index=False,
    encoding="utf-8-sig"
)


## 3. 해일 대피소에도 같은 규칙 적용
해일 대피소에도 동일한 주소 분해와 권역 필터를 적용해 전용 CSV를 만든다.


In [ ]:
ts = pd.read_csv(
    "data/tsunami_shelter_clean.csv",
    encoding="utf-8-sig"
)

ts.head()


In [ ]:
ts[["시도", "시군구"]] = (
    ts["주소"]
    .apply(lambda x: pd.Series(clean_korea_address(x)))
)


In [ ]:
ts = ts[ts["시도"].isin(target_region)]


In [ ]:
ts["시도"].value_counts()


In [ ]:
ts.to_csv(
    "data/tsunami_shelter_clean_2.csv",
    index=False,
    encoding="utf-8-sig"
)
